# M03 · Probabilités conditionnelles — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m03_probabilites_conditionnelles/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m03_probabilites_conditionnelles/03_notebook.ipynb`, créée par `python tools/start_chapter.py M03`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M03.1 à M03.11, M03.14, M03.15 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M03.19 | Jointe, marginales et conditionnelles à partir d'un tableau de comptages · `joint_from_counts, marginal, conditional` | 🔨 | ★ | 20 |
|  | M03.20 | Tester l'indépendance avec une tolérance · `are_independent` | 🔨 | ★ | 15 |
|  | M03.21 | L'AUC en comptant les paires · `auc_by_pairs` | 🔨 | ★★ | 20 |
|  | M03.22 | `sklearn.metrics.roc_auc_score` contre le comptage des paires, ex æquo compris | 📦 | ★ | 15 |
| B | M03.23 | Prédire : l'AUC après une transformation croissante des scores | 🔮 | ★ | 10 |
|  | M03.24 | Simuler pour estimer une probabilité conditionnelle | 🔬 | ★★ | 20 |
|  | M03.25 | La courbe ROC en escalier : une case par paire | 🎨 | ★ | 20 |
| C | M03.26 | Chasse à l'erreur : conditionnelle inversée, marginale oubliée | 🐛 | ★ | 15 |
|  | M03.27 | Une fonction qui refuse un tableau invalide (`ValueError`) | 🛠️ | ★ | 10 |
|  | M03.28 | Défi : retrouver un tableau de contingence à partir de quelques probabilités | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M03")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : tableaux de contingence, indépendance, probabilités totales, matrice de confusion, F-beta, balanced accuracy et MCC, prévalence, courbes ROC et precision-recall, AUC par les paires, score de Brier, seuils et taux par groupe.
- Écrire les cinq fonctions de `mymath/conditional.py` et les tester, puis utiliser `sklearn.metrics`.
- Prédire ce que voit l'AUC, estimer des conditionnelles par simulation, tracer une courbe ROC, chasser des erreurs de conditionnement, valider une entrée, reconstruire un tableau.

**Rappel express.** $P(A \mid B) = \frac{P(A \cap B)}{P(B)}$ ; $P(A \cap B) = P(A \mid B)\,P(B)$ ; indépendance : $P(A \cap B) = P(A)\,P(B)$ ; probabilités totales : $P(A) = \sum_k P(A \mid B_k)\,P(B_k)$ ; recall $= P(\hat{Y} = 1 \mid Y = 1)$, precision $= P(Y = 1 \mid \hat{Y} = 1)$ ; $\text{AUC} = P(S_+ > S_-) + \frac{1}{2}P(S_+ = S_-)$ ; Brier $= \frac{1}{n}\sum (p_i - y_i)^2$.

## Partie 0 · Vérifier tes exercices papier (✏️ M03.1 à M03.11, 🔗 M03.14 et M03.15)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"7/9"` ;
- une liste `[-1.5, 2.5]`, `True` / `False`, ou une lettre entre guillemets, `"B"`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M03.1 — Un tableau de contingence : jointes, marginales, conditionnelles**

In [ ]:
# Ex M03.1: the values you found on paper
answer_M03_1a = ...  # a) a number (2 decimals)
answer_M03_1b = ...  # b) a number (2 decimals)
answer_M03_1c = ...  # c) a number (3 decimals)
answer_M03_1d = ...  # d) a number (3 decimals)
answer_M03_1e = ...  # e) a number (3 decimals)
answer_M03_1f = ...  # f) a number (3 decimals)
answer_M03_1g = ...  # g) a list of 3 numbers (3 decimals)
answer_M03_1h = ...  # h) a number (3 decimals)

for letter, answer in zip("abcdefgh", [answer_M03_1a, answer_M03_1b, answer_M03_1c, answer_M03_1d, answer_M03_1e, answer_M03_1f, answer_M03_1g, answer_M03_1h]):
    wbm.check(f"M03.1{letter}", answer)

**Ex M03.2 — L'indépendance, vérifiée dans un tableau**

In [ ]:
# Ex M03.2: the values you found on paper
answer_M03_2a = ...  # a) a number (2 decimals)
answer_M03_2b = ...  # b) a number (2 decimals)
answer_M03_2c = ...  # c) True or False
answer_M03_2d = ...  # d) a number (2 decimals)
answer_M03_2e = ...  # e) a number (3 decimals)
answer_M03_2f = ...  # f) a number (2 decimals)
answer_M03_2g = ...  # g) True or False
answer_M03_2h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M03_2a, answer_M03_2b, answer_M03_2c, answer_M03_2d, answer_M03_2e, answer_M03_2f, answer_M03_2g, answer_M03_2h]):
    wbm.check(f"M03.2{letter}", answer)

**Ex M03.3 — Probabilités totales avec trois cas**

In [ ]:
# Ex M03.3: the values you found on paper
answer_M03_3a = ...  # a) a number (4 decimals)
answer_M03_3b = ...  # b) a number (3 decimals)
answer_M03_3c = ...  # c) a number (4 decimals)
answer_M03_3d = ...  # d) an integer
answer_M03_3e = ...  # e) a number (2 decimals)
answer_M03_3f = ...  # f) a number (4 decimals)
answer_M03_3g = ...  # g) a number (4 decimals)
answer_M03_3h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M03_3a, answer_M03_3b, answer_M03_3c, answer_M03_3d, answer_M03_3e, answer_M03_3f, answer_M03_3g, answer_M03_3h]):
    wbm.check(f"M03.3{letter}", answer)

**Ex M03.4 — Une matrice de confusion lue comme des probabilités**

In [ ]:
# Ex M03.4: the values you found on paper
answer_M03_4a = ...  # a) a number (2 decimals)
answer_M03_4b = ...  # b) a number (2 decimals)
answer_M03_4c = ...  # c) a number (2 decimals)
answer_M03_4d = ...  # d) a number (2 decimals)
answer_M03_4e = ...  # e) a number (3 decimals)
answer_M03_4f = ...  # f) a number (4 decimals)
answer_M03_4g = ...  # g) a number (2 decimals)
answer_M03_4h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M03_4a, answer_M03_4b, answer_M03_4c, answer_M03_4d, answer_M03_4e, answer_M03_4f, answer_M03_4g, answer_M03_4h]):
    wbm.check(f"M03.4{letter}", answer)

**Ex M03.5 — F-beta : quel β pour quel coût d'erreur ?**

In [ ]:
# Ex M03.5: the values you found on paper
answer_M03_5a = ...  # a) a number (4 decimals)
answer_M03_5b = ...  # b) a number (4 decimals)
answer_M03_5c = ...  # c) a number (4 decimals)
answer_M03_5d = ...  # d) an integer
answer_M03_5e = ...  # e) a number (2 decimals)
answer_M03_5f = ...  # f) a number (2 decimals)
answer_M03_5g = ...  # g) a number (4 decimals)
answer_M03_5h = ...  # h) a letter, as text

for letter, answer in zip("abcdefgh", [answer_M03_5a, answer_M03_5b, answer_M03_5c, answer_M03_5d, answer_M03_5e, answer_M03_5f, answer_M03_5g, answer_M03_5h]):
    wbm.check(f"M03.5{letter}", answer)

**Ex M03.6 — Balanced accuracy et MCC sur un problème déséquilibré**

In [ ]:
# Ex M03.6: the values you found on paper
answer_M03_6a = ...  # a) a number (2 decimals)
answer_M03_6b = ...  # b) a number (2 decimals)
answer_M03_6c = ...  # c) a number (2 decimals)
answer_M03_6d = ...  # d) a number (2 decimals)
answer_M03_6e = ...  # e) a number (4 decimals)
answer_M03_6f = ...  # f) a number (4 decimals)
answer_M03_6g = ...  # g) a number (3 decimals)
answer_M03_6h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M03_6a, answer_M03_6b, answer_M03_6c, answer_M03_6d, answer_M03_6e, answer_M03_6f, answer_M03_6g, answer_M03_6h]):
    wbm.check(f"M03.6{letter}", answer)

**Ex M03.7 — Changer de population : recalculer la precision quand la prévalence change**

In [ ]:
# Ex M03.7: the values you found on paper
answer_M03_7a = ...  # a) a number (4 decimals)
answer_M03_7b = ...  # b) a number (4 decimals)
answer_M03_7c = ...  # c) an integer
answer_M03_7d = ...  # d) an integer
answer_M03_7e = ...  # e) a number (3 decimals)
answer_M03_7f = ...  # f) a number (4 decimals)
answer_M03_7g = ...  # g) a number (4 decimals)
answer_M03_7h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M03_7a, answer_M03_7b, answer_M03_7c, answer_M03_7d, answer_M03_7e, answer_M03_7f, answer_M03_7g, answer_M03_7h]):
    wbm.check(f"M03.7{letter}", answer)

**Ex M03.8 — Une courbe ROC à la main, et son aire par les trapèzes**

In [ ]:
# Ex M03.8: the values you found on paper
answer_M03_8a = ...  # a) an integer
answer_M03_8b = ...  # b) a list [FPR, recall] (2 decimals)
answer_M03_8c = ...  # c) a list [FPR, recall] (2 decimals)
answer_M03_8d = ...  # d) a number (2 decimals)
answer_M03_8e = ...  # e) a number (2 decimals)
answer_M03_8f = ...  # f) a number (2 decimals)
answer_M03_8g = ...  # g) a number (4 decimals)
answer_M03_8h = ...  # h) a number (2 decimals)

for letter, answer in zip("abcdefgh", [answer_M03_8a, answer_M03_8b, answer_M03_8c, answer_M03_8d, answer_M03_8e, answer_M03_8f, answer_M03_8g, answer_M03_8h]):
    wbm.check(f"M03.8{letter}", answer)

**Ex M03.9 — L'AUC comme proportion de paires bien classées**

In [ ]:
# Ex M03.9: the values you found on paper
answer_M03_9a = ...  # a) an integer
answer_M03_9b = ...  # b) an integer
answer_M03_9c = ...  # c) an integer
answer_M03_9d = ...  # d) a number (2 decimals)
answer_M03_9e = ...  # e) a number (2 decimals)
answer_M03_9f = ...  # f) a number (4 decimals)
answer_M03_9g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M03_9a, answer_M03_9b, answer_M03_9c, answer_M03_9d, answer_M03_9e, answer_M03_9f, answer_M03_9g]):
    wbm.check(f"M03.9{letter}", answer)

**Ex M03.10 — Courbe precision-recall et average precision à la main**

In [ ]:
# Ex M03.10: the values you found on paper
answer_M03_10a = ...  # a) a list [precision, recall] (4 decimals)
answer_M03_10b = ...  # b) a list [precision, recall] (4 decimals)
answer_M03_10c = ...  # c) an irreducible fraction, as text
answer_M03_10d = ...  # d) a number (4 decimals)
answer_M03_10e = ...  # e) an irreducible fraction, as text
answer_M03_10f = ...  # f) a number (2 decimals)
answer_M03_10g = ...  # g) a number (2 decimals)
answer_M03_10h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M03_10a, answer_M03_10b, answer_M03_10c, answer_M03_10d, answer_M03_10e, answer_M03_10f, answer_M03_10g, answer_M03_10h]):
    wbm.check(f"M03.10{letter}", answer)

**Ex M03.11 — Score de Brier et calibration de huit prévisions**

In [ ]:
# Ex M03.11: the values you found on paper
answer_M03_11a = ...  # a) a number (3 decimals)
answer_M03_11b = ...  # b) a number (3 decimals)
answer_M03_11c = ...  # c) a number (3 decimals)
answer_M03_11d = ...  # d) a number (2 decimals)
answer_M03_11e = ...  # e) a number (2 decimals)
answer_M03_11f = ...  # f) a number (2 decimals)
answer_M03_11g = ...  # g) a number (2 decimals)
answer_M03_11h = ...  # h) a number (2 decimals)

for letter, answer in zip("abcdefgh", [answer_M03_11a, answer_M03_11b, answer_M03_11c, answer_M03_11d, answer_M03_11e, answer_M03_11f, answer_M03_11g, answer_M03_11h]):
    wbm.check(f"M03.11{letter}", answer)

**Ex M03.14 — Un seuil sur un score : trois seuils, trois matrices de confusion**

In [ ]:
# Ex M03.14: the values you found on paper
answer_M03_14a = ...  # a) an integer
answer_M03_14b = ...  # b) an integer
answer_M03_14c = ...  # c) an irreducible fraction, as text
answer_M03_14d = ...  # d) a number (2 decimals)
answer_M03_14e = ...  # e) a list [precision, recall] (3 decimals)
answer_M03_14f = ...  # f) a number (1 decimal)
answer_M03_14g = ...  # g) an integer
answer_M03_14h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_M03_14a, answer_M03_14b, answer_M03_14c, answer_M03_14d, answer_M03_14e, answer_M03_14f, answer_M03_14g, answer_M03_14h]):
    wbm.check(f"M03.14{letter}", answer)

**Ex M03.15 — Équité : les taux de faux positifs de deux groupes**

In [ ]:
# Ex M03.15: the values you found on paper
answer_M03_15a = ...  # a) a number (2 decimals)
answer_M03_15b = ...  # b) a number (2 decimals)
answer_M03_15c = ...  # c) a number (2 decimals)
answer_M03_15d = ...  # d) a number (4 decimals)
answer_M03_15e = ...  # e) a number (4 decimals)
answer_M03_15f = ...  # f) a number (2 decimals)
answer_M03_15g = ...  # g) True or False
answer_M03_15h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M03_15a, answer_M03_15b, answer_M03_15c, answer_M03_15d, answer_M03_15e, answer_M03_15f, answer_M03_15g, answer_M03_15h]):
    wbm.check(f"M03.15{letter}", answer)

## Partie A · Ta librairie : loi jointe, marginales, conditionnelles, indépendance et AUC

Fiche §1, §2 et §7. Les cinq fonctions de `mymath/conditional.py`, puis l'AUC et l'average precision avec scikit-learn.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import average_precision_score, balanced_accuracy_score, roc_auc_score, roc_curve

from wbm.nbtools import checked, drawn, run_pytest_group, verdict


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M03.19 — Jointe, marginales et conditionnelles à partir d'un tableau de comptages · `joint_from_counts, marginal, conditional` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le passage d'un tableau de comptages à la loi jointe, ses marginales et ses lois conditionnelles, avec leurs cas d'erreur.  
**Prérequis :** fiche §1 et §2 · **mymath :** `conditional.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M02c). Ouvre ta copie `mon_travail/mymath/conditional.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy est permis (importe-le en haut du fichier), scikit-learn non : les tests comparent tes résultats à lui, et un test vérifie que tu ne t'en sers pas. Une loi jointe est un tableau à deux dimensions : la ligne `i` est la valeur $x_i$ de $X$, la colonne `j` la valeur $y_j$ de $Y$. Les tableaux renvoyés sont des arrays NumPy de `float`, les nombres des `float` ou des `bool` de Python.

Écris, dans cet ordre (fiche §1 et §2) :
- `joint_from_counts(counts)` : chaque comptage divisé par le total ; une `ValueError` si le tableau n'a pas deux dimensions, a un comptage négatif, ou ne contient que des zéros ;
- `marginal(joint, axis)` : la convention de `np.sum(joint, axis=axis)`, donc `axis=0` donne la loi de $Y$ (une probabilité par **colonne**) et `axis=1` celle de $X$ (une par **ligne**) ; une `ValueError` si `joint` n'est pas une loi jointe valide (des cases positives de somme 1, à $10^{-9}$ près) ou si `axis` ne vaut ni 0 ni 1 ;
- `conditional(joint, given_row=None, given_column=None)` : la loi de $Y$ sachant $X = x_i$ (la ligne `i` divisée par sa somme), ou celle de $X$ sachant $Y = y_j$ ; une `ValueError` si l'on donne les deux indices ou aucun, si l'indice sort du tableau, ou si l'événement de la condition a une probabilité nulle. Un indice est dans le tableau s'il va de 0 au nombre de lignes (ou de colonnes) moins 1.

Une fonction auxiliaire `_check_joint(joint)`, qui convertit et vérifie une loi jointe, évite d'écrire trois fois les mêmes vérifications.

La vérification utilise `TABLE_19`, les 200 utilisateurs d'une application de cuisine, selon leur formule (essai gratuit ou abonnement payant, en lignes) et leur usage du minuteur intégré (jamais, parfois, souvent, en colonnes). Elle calcule :
- a) la loi jointe, `joint_from_counts(TABLE_19)` (3 décimales) ;
- b) sa marginale avec `axis=0` (3 décimales) ;
- c) sa marginale avec `axis=1` (3 décimales) ;
- d) la loi de l'usage sachant « essai gratuit », `given_row=0` (3 décimales) ;
- e) la loi de la formule sachant « souvent », `given_column=2` (3 décimales) ;
- f) le nom de l'erreur levée par `joint_from_counts([[1, -2], [3, 4]])` ;
- g) le nom de l'erreur levée par `conditional(joint, 0, 1)`, avec les deux indices ;
- h) le nom de l'erreur levée par `marginal([[0.5, 0.6], [0.1, 0.2]], 0)`.

Dans ta copie : refais a), d) et e) à la main sur le tableau. Lequel des deux, b) ou c), est la loi de la formule ?

In [ ]:
TABLE_19 = [[45, 30, 25], [15, 45, 40]]   # rows: free trial, paid plan; columns: never, sometimes, often

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M03")  # reload your saved file

In [ ]:
with checked("M03.19a"):
    wbm.check("M03.19a", mymath.conditional.joint_from_counts(TABLE_19), computed=True)
joint_19 = np.array(TABLE_19) / np.sum(TABLE_19)     # the right joint law, so that b) to h) do not depend on a)
with checked("M03.19b"):
    wbm.check("M03.19b", mymath.conditional.marginal(joint_19, 0), computed=True)
with checked("M03.19c"):
    wbm.check("M03.19c", mymath.conditional.marginal(joint_19, 1), computed=True)
with checked("M03.19d"):
    wbm.check("M03.19d", mymath.conditional.conditional(joint_19, given_row=0), computed=True)
with checked("M03.19e"):
    wbm.check("M03.19e", mymath.conditional.conditional(joint_19, given_column=2), computed=True)
with checked("M03.19f"):
    wbm.check("M03.19f", raised(mymath.conditional.joint_from_counts, [[1, -2], [3, 4]]), computed=True)
with checked("M03.19g"):
    wbm.check("M03.19g", raised(mymath.conditional.conditional, joint_19, 0, 1), computed=True)
with checked("M03.19h"):
    wbm.check("M03.19h", raised(mymath.conditional.marginal, [[0.5, 0.6], [0.1, 0.2]], 0), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m03_conditional.py", "test_joint_from_counts_ or test_marginal_ or test_conditional_")

### Ex M03.20 — Tester l'indépendance avec une tolérance · `are_independent` 🔨 ★ ⏱️ 15 min
**Objectif :** décider si deux variables sont indépendantes en comparant chaque case de leur loi jointe au produit de ses marginales, avec une tolérance.  
**Prérequis :** fiche §2 ; Ex M03.19 · **mymath :** `conditional.py` · **Parcours :** C

Écris `are_independent(joint, tol=1e-9)` (fiche §2) : `True` si **chaque** case de la loi jointe diffère du produit de ses deux marginales d'au plus `tol`, `False` sinon. `np.outer(a, b)` construit d'un coup le tableau de tous les produits `a[i] * b[j]`. Le résultat est un `bool` de Python (`bool(...)` convertit un booléen de NumPy) ; une loi jointe invalide lève une `ValueError`. Ta fonction peut appeler `marginal`.

La vérification calcule :
- a) `are_independent` pour la loi jointe de `TABLE_19` ;
- b) pour la loi `np.outer([0.3, 0.7], [0.2, 0.5, 0.3])`, construite indépendante par le calcul en flottants ;
- c) pour la loi `TABLE_20`, issue d'un échantillon de 1 000 personnes, avec la tolérance par défaut ;
- d) pour `TABLE_20` avec `tol=0.005` ;
- e) pour la loi `[[0.1, 0.2, 0.2], [0.1, 0.1, 0.3]]` ;
- f) `isinstance(résultat, bool)` : le résultat est-il un `bool` de Python ? (Le nom du type ne suffit pas : depuis NumPy 2, celui d'un booléen de NumPy s'écrit aussi `bool`.)
- g) le nom de l'erreur levée pour `[[0.5, 0.5], [0.5, 0.5]]` ;
- h) `are_independent` pour `TABLE_20H`, dont chaque case s'écarte de $2 	imes 10^{-6}$ du produit de ses marges, avec la tolérance par défaut.

Dans ta copie : pour la loi de e), compare la première case au produit de ses marges, puis les autres. Pour `TABLE_20`, de combien chaque case s'écarte-t-elle du produit ? Sur des données réelles, pourquoi ne trouve-t-on jamais une indépendance exacte, et que faudrait-il pour décider (fiche §2) ?

In [ ]:
TABLE_20 = np.array([[61, 139], [239, 561]]) / 1000   # a sample of 1 000 people: a joint law estimated from counts
TABLE_20H = np.outer([0.5, 0.5], [0.5, 0.5]) + np.array([[2e-6, -2e-6], [-2e-6, 2e-6]])   # gaps of 2e-6

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M03")  # reload your saved file

In [ ]:
joint_19 = np.array(TABLE_19) / np.sum(TABLE_19)
with checked("M03.20a"):
    wbm.check("M03.20a", mymath.conditional.are_independent(joint_19), computed=True)
with checked("M03.20b"):
    wbm.check("M03.20b", mymath.conditional.are_independent(np.outer([0.3, 0.7], [0.2, 0.5, 0.3])), computed=True)
with checked("M03.20c"):
    wbm.check("M03.20c", mymath.conditional.are_independent(TABLE_20), computed=True)
with checked("M03.20d"):
    wbm.check("M03.20d", mymath.conditional.are_independent(TABLE_20, 0.005), computed=True)
with checked("M03.20e"):
    wbm.check("M03.20e", mymath.conditional.are_independent([[0.1, 0.2, 0.2], [0.1, 0.1, 0.3]]), computed=True)
with checked("M03.20f"):
    wbm.check("M03.20f", isinstance(mymath.conditional.are_independent(joint_19), bool), computed=True)
with checked("M03.20g"):
    wbm.check("M03.20g", raised(mymath.conditional.are_independent, [[0.5, 0.5], [0.5, 0.5]]), computed=True)
with checked("M03.20h"):
    wbm.check("M03.20h", mymath.conditional.are_independent(TABLE_20H), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m03_conditional.py", "test_are_independent_")

### Ex M03.21 — L'AUC en comptant les paires · `auc_by_pairs` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer l'AUC comme la proportion des paires (positif, négatif) bien classées, ex æquo compris, et la comparer à scikit-learn.  
**Prérequis :** fiche §7 · **mymath :** `conditional.py` · **Parcours :** C, E

Écris `auc_by_pairs(scores_pos, scores_neg)` (fiche §7) : chaque paire (positif, négatif) compte 1 si le positif a le score le plus élevé, $\frac{1}{2}$ en cas d'égalité, 0 sinon, et l'on divise le total par le nombre de paires. Une liste vide lève une `ValueError` ; le résultat est un `float` de Python. Deux boucles `for` imbriquées conviennent ; le **broadcasting** de NumPy fait mieux : `pos[:, None] - neg[None, :]` est le tableau de toutes les différences, une ligne par positif et une colonne par négatif.

La vérification calcule (4 décimales) :
- a) `auc_by_pairs([0.8, 0.6, 0.55], [0.7, 0.3])` ;
- b) `auc_by_pairs([0.5, 0.5, 0.9], [0.5, 0.1])`, avec des ex æquo ;
- c) `auc_by_pairs([3, 4], [1, 2])` ;
- d) `auc_by_pairs([1, 2], [3, 4])` ;
- e) le nom de l'erreur levée par `auc_by_pairs([], [0.2])` ;
- f) le nom du type du résultat, en texte ;
- g) l'AUC des 2 000 positifs et 3 000 négatifs de la cellule fournie (6 millions de paires).

Dans ta copie : refais a) et b) à la main, en écrivant le tableau des paires. Combien de temps prend g) avec ta méthode ? La valeur théorique de g) est $P(S_+ > S_-)$ pour $S_+ \sim \mathcal{N}(1, 1)$ et $S_- \sim \mathcal{N}(0, 1)$ indépendantes : avec M02b et M02c, montre qu'elle vaut $\Phi\big(\frac{1}{\sqrt{2}}\big) \approx 0{,}760$.

In [ ]:
rng_21 = np.random.default_rng(21)
POS_21 = rng_21.normal(1.0, 1.0, 2000)     # scores of 2 000 positive examples
NEG_21 = rng_21.normal(0.0, 1.0, 3000)     # scores of 3 000 negative examples

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M03")  # reload your saved file

In [ ]:
with checked("M03.21a"):
    wbm.check("M03.21a", mymath.conditional.auc_by_pairs([0.8, 0.6, 0.55], [0.7, 0.3]), computed=True)
with checked("M03.21b"):
    wbm.check("M03.21b", mymath.conditional.auc_by_pairs([0.5, 0.5, 0.9], [0.5, 0.1]), computed=True)
with checked("M03.21c"):
    wbm.check("M03.21c", mymath.conditional.auc_by_pairs([3, 4], [1, 2]), computed=True)
with checked("M03.21d"):
    wbm.check("M03.21d", mymath.conditional.auc_by_pairs([1, 2], [3, 4]), computed=True)
with checked("M03.21e"):
    wbm.check("M03.21e", raised(mymath.conditional.auc_by_pairs, [], [0.2]), computed=True)
with checked("M03.21f"):
    wbm.check("M03.21f", type(mymath.conditional.auc_by_pairs([0.8], [0.3])).__name__, computed=True)
with checked("M03.21g"):
    wbm.check("M03.21g", mymath.conditional.auc_by_pairs(POS_21, NEG_21), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m03_conditional.py", "test_auc_by_pairs_")

### Ex M03.22 — `sklearn.metrics.roc_auc_score` contre le comptage des paires, ex æquo compris 📦 ★ ⏱️ 15 min
**Objectif :** calculer avec scikit-learn l'AUC, la courbe ROC et l'average precision de scores arrondis, et relier une AUC calculée sur des prédictions 0/1 à la balanced accuracy.  
**Prérequis :** fiche §5 et §7 · **Parcours :** C

Les fonctions de `sklearn.metrics` prennent toujours **les labels vrais d'abord**, puis les scores : `roc_auc_score(y_true, y_score)`, `roc_curve(y_true, y_score)` (qui renvoie `fpr, tpr, thresholds`), `average_precision_score(y_true, y_score)`, `balanced_accuracy_score(y_true, y_pred)`. Lis `help(roc_curve)` avant de commencer.

La cellule fournie définit `Y_22` et `S_22` : 12 exemples, dont plusieurs scores égaux. Calcule :
- a) l'AUC, `roc_auc_score` (4 décimales) ;
- b) l'AUC des scores opposés, `-S_22` (4 décimales) ;
- c) l'AUC des **prédictions** 0/1 au seuil 0,5, `(S_22 >= 0.5).astype(int)`, données à la place des scores (4 décimales) ;
- d) la balanced accuracy de ces mêmes prédictions (4 décimales) ;
- e) le nombre de seuils renvoyés par `roc_curve(Y_22, S_22)` (un entier) ;
- f) l'average precision (4 décimales).

Dans ta copie : retrouve a) en comptant les paires (ta fonction `auc_by_pairs` le fait aussi). Pourquoi c) et d) sont-ils égaux ? (Trace la courbe ROC d'un classifieur qui ne donne que 0 ou 1 : combien a-t-elle de points ?) Que vaut le premier seuil de e), et à quel point de la courbe correspond-il ?

In [ ]:
Y_22 = np.array([1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1])                                # 6 positives, 6 negatives
S_22 = np.array([0.8, 0.8, 0.9, 0.4, 0.4, 0.5, 0.2, 0.6, 0.1, 0.15, 0.4, 0.8])     # scores rounded to 0.1 or 0.05

In [ ]:
answer_M03_22a = ...  # a) roc_auc_score(...)
answer_M03_22b = ...  # b) with -S_22
answer_M03_22c = ...  # c) on the 0/1 predictions at the threshold 0.5
answer_M03_22d = ...  # d) balanced_accuracy_score(...)
answer_M03_22e = ...  # e) an integer
answer_M03_22f = ...  # f) average_precision_score(...)

In [ ]:
for letter, answer in zip("abcdef", [answer_M03_22a, answer_M03_22b, answer_M03_22c, answer_M03_22d,
                                     answer_M03_22e, answer_M03_22f]):
    wbm.check(f"M03.22{letter}", answer)

## Partie B · Une prédiction, une simulation et une figure

Fiche §2, §7 et §8. Ce que l'AUC voit et ne voit pas, des probabilités conditionnelles estimées par simulation, la courbe ROC en escalier.

### Ex M03.23 — Prédire : l'AUC après une transformation croissante des scores 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir l'effet d'une transformation des scores sur l'AUC et sur le score de Brier, puis le vérifier.  
**Prérequis :** fiche §7 et §8 · **Parcours :** C, E

Un modèle donne à 400 exemples un score $s$ entre 0 et 1. On remplace ces scores par une transformation, et l'on recalcule l'AUC et le score de Brier. Choisis, pour chaque question :
- a) l'AUC des scores $10\,s$ : (1) la même ; (2) multipliée par 10 ; (3) elle change, dans un sens imprévisible ; (4) elle vaut 1 ;
- b) l'AUC des scores $s^2$ : (1) la même ; (2) élevée au carré ; (3) plus petite ; (4) plus grande ;
- c) l'AUC des scores $1 - s$ : (1) la même ; (2) $1 - \text{AUC}$ ; (3) 0,5 ; (4) 0 ;
- d) l'AUC quand chaque score est remplacé par son **rang** (1 pour le plus petit, 400 pour le plus grand) : (1) la même ; (2) elle change ; (3) elle vaut 0,5 ; (4) elle dépend du nombre d'exemples ;
- e) le score de Brier des scores $s^2$, comparé à celui des scores $s$ : (1) le même ; (2) différent.

**Avant d'exécuter quoi que ce soit**, écris tes cinq prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M03_23a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M03_23b = ...  # b) 1, 2, 3 or 4
prediction_M03_23c = ...  # c) 1, 2, 3 or 4
prediction_M03_23d = ...  # d) 1, 2, 3 or 4
prediction_M03_23e = ...  # e) 1 or 2

In [ ]:
for letter, prediction in zip("abcde", [prediction_M03_23a, prediction_M03_23b, prediction_M03_23c,
                                       prediction_M03_23d, prediction_M03_23e]):
    wbm.check(f"M03.23{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes cinq prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M03_23a, prediction_M03_23b, prediction_M03_23c, prediction_M03_23d, prediction_M03_23e]):
    print("⏳ Écris d'abord tes cinq prédictions, puis exécute à nouveau.")
else:
    rng_23 = np.random.default_rng(23)
    y_23 = rng_23.random(400) < 0.3                                          # 400 examples, about 30 % positive
    s_23 = np.where(y_23, rng_23.beta(5, 3, 400), rng_23.beta(2, 5, 400))    # scores between 0 and 1
    ranks_23 = np.argsort(np.argsort(s_23)) + 1                              # 1 for the smallest score, 400 for the largest
    for name, scores in [("s", s_23), ("10 s", 10 * s_23), ("s²", s_23 ** 2), ("1 − s", 1 - s_23), ("rang", ranks_23)]:
        print(f"{name:>6} : AUC {roc_auc_score(y_23, scores):.4f}")
    for name, scores in [("s", s_23), ("s²", s_23 ** 2)]:
        print(f"{name:>6} : score de Brier {np.mean((scores - y_23) ** 2):.4f}")

**Après l'expérience**, dans ta copie : explique chaque résultat avec la lecture de l'AUC comme proportion de paires (fiche §7). Pourquoi le score de Brier, lui, change-t-il ? Que dit cette expérience de ce que mesure l'AUC, et de ce qu'elle ne mesure pas ?

### Ex M03.24 — Simuler pour estimer une probabilité conditionnelle 🔬 ★★ ⏱️ 20 min
**Objectif :** estimer des probabilités conditionnelles par la fréquence dans des tirages simulés, et les comparer aux valeurs exactes.  
**Prérequis :** fiche §2 ; R8 · **Parcours :** C

Pour estimer $P(A \mid B)$ par simulation, on ne garde que les tirages où $B$ s'est produit, et l'on y compte la proportion de ceux où $A$ s'est produit aussi : c'est la définition du §2, appliquée aux fréquences. La cellule fournie lance 100 000 fois deux dés à six faces (`DICE_24`, une ligne par lancer, une colonne par dé).

Écris `conditional_frequency(event, given)`, qui reçoit deux tableaux de booléens de même longueur (un par tirage) et renvoie la fréquence de `event` parmi les tirages où `given` est vrai, en `float`. Si `given` n'est jamais vrai, la fréquence n'a pas de sens : lève une `ValueError`. Avec NumPy, `event[given]` garde les seules valeurs de `event` où `given` est vrai.

La vérification appelle ta fonction et calcule (3 décimales) :
- a) la fréquence de « la somme vaut au moins 10 » sachant « au moins un 6 » ;
- b) la fréquence de « au moins un 6 » sachant « la somme vaut au moins 10 » ;
- c) la fréquence de « le premier dé donne 6 » sachant « la somme vaut au moins 10 » ;
- d) la fréquence de « le premier dé est pair » sachant « la somme est paire » ;
- e) la fréquence de « le premier dé est pair », sans condition (`given` toujours vrai) ;
- f) le nom de l'erreur levée par `conditional_frequency(somme >= 10, somme > 12)` ;
- g) le produit a) × (fréquence de « au moins un 6 »), comparé à la fréquence de l'intersection (la règle du produit).

Dans ta copie : calcule les valeurs exactes de a) à e) en comptant les 36 issues équiprobables (a) vaut $\frac{5}{11}$). Pourquoi a) et b) sont-ils si différents ? Que montrent d) et e) ? L'écart entre une fréquence et sa valeur exacte est-il raisonnable, compte tenu de l'erreur type d'une proportion (M02c §7) et du **nombre** de tirages où la condition est vraie ?

In [ ]:
DICE_24 = np.random.default_rng(2075).integers(1, 7, (100_000, 2))   # 100 000 throws of two dice, one per row

In [ ]:
def conditional_frequency(event, given):
    """The frequency of event among the draws where given is True (two arrays of booleans, one value per draw)."""
    raise NotImplementedError("ne garde que les tirages où given est vrai, puis la proportion de event parmi eux")

In [ ]:
total_24 = DICE_24.sum(axis=1)
six_24 = (DICE_24 == 6).any(axis=1)              # at least one 6
with checked("M03.24a"):
    wbm.check("M03.24a", conditional_frequency(total_24 >= 10, six_24), computed=True)
with checked("M03.24b"):
    wbm.check("M03.24b", conditional_frequency(six_24, total_24 >= 10), computed=True)
with checked("M03.24c"):
    wbm.check("M03.24c", conditional_frequency(DICE_24[:, 0] == 6, total_24 >= 10), computed=True)
with checked("M03.24d"):
    wbm.check("M03.24d", conditional_frequency(DICE_24[:, 0] % 2 == 0, total_24 % 2 == 0), computed=True)
with checked("M03.24e"):
    wbm.check("M03.24e", conditional_frequency(DICE_24[:, 0] % 2 == 0, np.ones(len(DICE_24), dtype=bool)),
              computed=True)
with checked("M03.24f"):
    wbm.check("M03.24f", raised(conditional_frequency, total_24 >= 10, total_24 > 12), computed=True)
with checked("M03.24g"):
    wbm.check("M03.24g", conditional_frequency(total_24 >= 10, six_24) * float(np.mean(six_24)), computed=True)

### Ex M03.25 — La courbe ROC en escalier : une case par paire 🎨 ★ ⏱️ 20 min
**Objectif :** calculer les sommets d'une courbe ROC, ex æquo compris, et la tracer avec la diagonale du hasard et son AUC.  
**Prérequis :** fiche §7 ; R0 (matplotlib) · **Parcours :** C

Écris deux fonctions :
- `roc_points(scores_pos, scores_neg)` : la liste des sommets de la courbe ROC, des couples `(fpr, tpr)` : d'abord $(0, 0)$, puis un point pour chaque score **distinct**, du plus grand au plus petit, avec la règle « positif si score $\ge$ seuil » ; le dernier point est $(1, 1)$. Un seuil qui fait passer à la fois des positifs et des négatifs (des ex æquo) fait avancer les deux coordonnées d'un coup ;
- `draw_roc(ax, scores_pos, scores_neg)` : dessine, dans l'axe `ax`, la courbe qui relie ces points par des segments (`ax.plot`), la diagonale du hasard en pointillés, les noms des deux axes, et un titre qui contient l'AUC (ta fonction `mymath.conditional.auc_by_pairs`, ou `roc_auc_score`).

Puis trace la courbe des scores `POS_25` et `NEG_25` de la cellule fournie. Si tu le veux, ajoute en fond la grille des $8 \times 6$ paires (`ax.set_xticks(np.linspace(0, 1, 7))`, `ax.set_yticks(np.linspace(0, 1, 9))`, `ax.grid(True)`), comme la figure de la fiche.

Dans ta copie : compte les cases de la grille sous la courbe ; retrouves-tu l'AUC ? Où se trouve le segment en diagonale, et à quelles paires correspond-il ?

In [ ]:
POS_25 = [0.93, 0.88, 0.71, 0.71, 0.64, 0.52, 0.47, 0.38]     # 8 positive examples
NEG_25 = [0.82, 0.71, 0.55, 0.40, 0.35, 0.21]                   # 6 negative examples (one tie with two positives)

In [ ]:
def roc_points(scores_pos, scores_neg):
    """The vertices (fpr, tpr) of the ROC curve: (0, 0), then one point per distinct score, from the largest to the
    smallest, with the rule "positive if score ≥ threshold"."""
    raise NotImplementedError("les scores distincts triés par ordre décroissant, puis deux proportions par seuil")


def draw_roc(ax, scores_pos, scores_neg):
    """In ax: the ROC curve (segments between the points), the dashed chance diagonal, axis names, the AUC in the
    title."""
    raise NotImplementedError("ax.plot des points de roc_points, ax.plot([0, 1], [0, 1], \"--\"), set_xlabel, set_title")

In [ ]:
with checked("M03.25 figure"):
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    try:
        created = drawn(draw_roc, ax, POS_25, NEG_25)
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M03.25 : draw_roc ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans l'axe reçu.")
    else:
        plt.show()

**Vérifications automatiques** (sur d'autres scores, avec des ex æquo ; la figure elle-même se juge à l'œil).

In [ ]:
with checked("M03.25 vérifications"):
    problems = []
    pos_t, neg_t = [0.9, 0.6, 0.6, 0.3, 0.1], [0.6, 0.4, 0.3, 0.2]       # another example, with ties
    y_t = np.r_[np.ones(len(pos_t)), np.zeros(len(neg_t))]
    fpr_t, tpr_t, _ = roc_curve(y_t, np.r_[pos_t, neg_t], drop_intermediate=False)
    expected = [(float(f), float(t)) for f, t in zip(fpr_t, tpr_t)]
    result = list(roc_points(pos_t, neg_t))
    if (len(result) == 2 and all(np.ndim(part) == 1 and len(part) == len(expected) for part in result)):
        problems.append("roc_points renvoie deux listes (fpr, tpr), comme roc_curve : la docstring demande une liste de "
                        "couples (fpr, tpr), par exemple list(zip(fpr, tpr))")
    else:
        points = [tuple(float(v) for v in point) for point in result]
        if len(points) == len(pos_t) + len(neg_t) + 1:
            problems.append("roc_points : un point par score distinct, pas un par exemple ; des ex æquo passent le "
                            "seuil ensemble et font avancer les deux coordonnées d'un coup")
        elif len(points) != len(expected) or any(abs(a - c) > 1e-9 or abs(b - d) > 1e-9
                                                 for (a, b), (c, d) in zip(points, expected)):
            problems.append("roc_points : un point (0, 0), puis un point par score distinct, du plus grand au plus "
                            "petit, chacun (FPR, recall) avec la règle « score ≥ seuil »")
    fig, ax = plt.subplots(figsize=(4, 4))
    try:
        created = drawn(draw_roc, ax, pos_t, neg_t)
    finally:
        plt.close(fig)                                     # only the checks below are shown here
    if created:
        problems.append("draw_roc ouvre sa propre figure : dessine dans l'axe reçu")
    else:
        lines = [line for line in ax.get_lines() if len(line.get_xdata()) >= 3]
        diagonals = [line for line in ax.get_lines()
                     if np.allclose(np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float))
                     and min(line.get_xdata()) <= 0 and max(line.get_xdata()) >= 1]
        curve = None
        for line in lines:
            xy = list(zip(np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)))
            if all(any(abs(x - a) < 1e-9 and abs(y - b) < 1e-9 for x, y in xy) for a, b in expected):
                curve = line
        if curve is None:
            problems.append("la courbe ROC doit passer par tous ses points (FPR, recall) : trace roc_points avec ax.plot")
        else:
            if curve.get_drawstyle() != "default":
                problems.append("relie les points par des segments (ax.plot) : un ex æquo donne un segment oblique, "
                                "que ax.step dessinerait en marche")
            if curve.get_linestyle() in ("None", "none", "", " "):
                problems.append("relie les points par des segments : ax.plot(fpr, tpr), pas des points seuls (le format « o »)")
            xs, ys = np.asarray(curve.get_xdata(), dtype=float), np.asarray(curve.get_ydata(), dtype=float)
            area = float(np.sum((xs[1:] - xs[:-1]) * (ys[1:] + ys[:-1]) / 2))
            if abs(area - roc_auc_score(y_t, np.r_[pos_t, neg_t])) > 1e-9:
                problems.append("l'aire sous la courbe tracée doit être l'AUC : relie dans l'ordre, de (0, 0) à (1, 1), "
                                "les seuls points de roc_points (un par score distinct)")
        if not diagonals:
            problems.append("il manque la diagonale du hasard, de (0, 0) à (1, 1)")
        if not ax.get_xlabel() or not ax.get_ylabel():
            problems.append("nomme les deux axes (taux de faux positifs, recall)")
        if "AUC" not in ax.get_title():
            problems.append("le titre doit donner l'AUC")
    if problems:
        print("❌ Ex M03.25 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M03.25 : une courbe ROC juste, ex æquo compris, avec la diagonale, les axes nommés et l'AUC.")

## Partie C · Erreurs classiques, validation et défi

Fiche §1 à §3. Quatre erreurs de conditionnement à trouver, une fonction qui refuse les entrées invalides, puis un tableau à reconstruire.

### Ex M03.26 — Chasse à l'erreur : conditionnelle inversée, marginale oubliée 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques du calcul de probabilités conditionnelles, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §2 et §3 · **Parcours :** C

Une plateforme de cours en ligne connaît le niveau de ses inscrits et la probabilité qu'ils terminent un cours selon leur niveau :

```python
p_level = {"débutant": 0.5, "intermédiaire": 0.3, "avancé": 0.2}       # P(niveau)
p_done = {"débutant": 0.2, "intermédiaire": 0.4, "avancé": 0.7}        # P(termine | niveau)
```

Un analyste a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle veut calculer pour un inscrit tiré au hasard. Quatre lignes sont fausses. **Sans exécuter de code**, trouve-les.

```python
p_A = (0.2 + 0.4 + 0.7) / 3          # (A) P(termine)
p_B = 0.7 * 0.2                      # (B) P(avancé et termine)
p_C = 0.7                            # (C) P(avancé | termine)
p_D = 0.5 * 0.8                      # (D) P(débutant | ne termine pas)
p_E = 0.4 + 0.7                      # (E) P(termine | pas débutant)
p_F = 1 - 0.7                        # (F) P(ne termine pas | avancé)
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 3 décimales).

Dans ta copie : pour chaque ligne fausse, quelle confusion a été faite ? Une des lignes fausses donne une valeur impossible pour une probabilité : laquelle ?

In [ ]:
answer_M03_26a = ...  # a) letters, as text
answer_M03_26b = ...  # b) a list of 6 values (3 decimals)

In [ ]:
wbm.check("M03.26a", answer_M03_26a)
wbm.check("M03.26b", answer_M03_26b)

### Ex M03.27 — Une fonction qui refuse un tableau invalide (`ValueError`) 🛠️ ★ ⏱️ 10 min
**Objectif :** écrire une fonction de validation qui refuse les entrées invalides par une `ValueError` au message clair, sans laisser passer le cas piège des NaN.  
**Prérequis :** fiche §1 ; Ex M03.19 · **Parcours :** C

Une fonction qui reçoit des données doit refuser ce qui n'a pas de sens, **tôt** et avec un message clair, plutôt que de renvoyer un résultat faux plus loin. En Python, l'erreur habituelle pour une valeur invalide est la `ValueError`.

Écris `check_joint(table, tol=1e-9)`, qui renvoie le tableau converti en array NumPy de `float` s'il est une loi jointe valide, et lève sinon une `ValueError` dont le message dit **quelle règle** est violée :
- le tableau doit avoir deux dimensions (et au moins une case) ;
- toutes ses cases doivent être positives ou nulles ;
- leur somme doit valoir 1, à `tol` près.

Attention au piège : une case `nan` (*not a number*, une valeur manquante dans un calcul) rend la somme `nan`, et **toute** comparaison avec `nan` est fausse : `abs(nan - 1) > tol` vaut `False`, comme `nan < 0`. `np.isfinite` repère les cases qui ne sont pas des nombres finis.

La vérification essaie huit tableaux, valides ou non.

Dans ta copie : dans quel ordre as-tu écrit les vérifications, et pourquoi l'ordre compte-t-il (que se passe-t-il si l'on additionne les cases d'un tableau vide) ? Ta fonction `_check_joint` de `mymath/conditional.py` résiste-t-elle aux NaN ?

In [ ]:
def check_joint(table, tol=1e-9):
    """Return table as a 2-D float array if it is a valid joint law; otherwise raise ValueError with a message that
    says which rule is broken (two dimensions, non-negative finite entries, sum 1 up to tol)."""
    raise NotImplementedError("np.asarray(table, dtype=float), puis une vérification par règle, chacune avec son message")

In [ ]:
with checked("M03.27 vérifications"):
    problems = []
    cases = [("une loi 2 × 2 valide", [[0.1, 0.2], [0.3, 0.4]], True),
             ("une loi d'entiers valide", [[0, 1], [0, 0]], True),
             ("une somme de 1 à 1e-12 près", [[0.1, 0.2], [0.3, 0.4 + 1e-12]], True),
             ("un tableau à une dimension", [0.5, 0.5], False),
             ("une case négative (somme 1)", [[0.6, -0.1], [0.3, 0.2]], False),
             ("une somme de 1,2", [[0.3, 0.3], [0.3, 0.3]], False),
             ("une case NaN", [[float("nan"), 0.5], [0.25, 0.25]], False),
             ("un tableau vide", [[]], False)]
    messages = {}
    for name, table, valid in cases:
        try:
            result = check_joint(table)
        except NotImplementedError:
            raise
        except ValueError as exc:
            if valid:
                problems.append(f"{name} est refusée à tort ({exc})")
            else:
                messages[name] = str(exc)
                if not str(exc).strip():
                    problems.append(f"{name} : la ValueError doit avoir un message qui dit quelle règle est violée")
            continue
        except Exception as exc:  # noqa: BLE001
            problems.append(f"{name} : lève {type(exc).__name__} ; la docstring demande une ValueError")
            continue
        if not valid:
            problems.append(f"{name} est acceptée à tort" + (" (abs(nan − 1) > 1e-9 est False : une comparaison avec "
                                                            "NaN est toujours fausse)" if "NaN" in name else ""))
        elif not (isinstance(result, np.ndarray) and result.dtype == float and result.ndim == 2):
            problems.append(f"{name} : renvoie un array NumPy de float à deux dimensions")
    rules = [messages.get(name) for name in ("un tableau à une dimension", "une case négative (somme 1)",
                                             "une somme de 1,2")]
    if all(rules) and len(set(rules)) < 3:
        problems.append("les trois règles (deux dimensions, cases positives, somme 1) doivent avoir trois messages "
                        "différents : un message doit dire ce qui ne va pas")
    if problems:
        print("❌ Ex M03.27 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M03.27 : les tableaux valides passent, les invalides lèvent une ValueError qui dit pourquoi.")

### Ex M03.28 — Défi : retrouver un tableau de contingence à partir de quelques probabilités 🏆 ★★★ ⏱️ 25 min
**Objectif :** reconstruire un tableau de contingence à partir de marges et de probabilités conditionnelles, par le calcul ou par une recherche.  
**Prérequis :** fiche §1 à §3 · **Parcours :** C

Une boutique a perdu le tableau de ses 600 visites d'hier, classées selon le canal (appli ou site, en lignes) et le résultat (achat, panier abandonné, ou rien, en colonnes). Il lui reste ces informations, pour une visite tirée au hasard :
1. il y a eu 600 visites ;
2. $P(\text{appli}) = 0{,}4$ ;
3. $P(\text{achat}) = 0{,}5$ ;
4. $P(\text{achat} \mid \text{appli}) = 0{,}625$ ;
5. $P(\text{appli} \mid \text{abandon}) = \frac{1}{6}$ ;
6. $P(\text{rien} \mid \text{site}) = \frac{1}{6}$.

Retrouve le tableau des **effectifs** : `TABLE_28`, une liste de 2 listes de 3 entiers, `[[appli-achat, appli-abandon, appli-rien], [site-achat, site-abandon, site-rien]]`. Deux chemins sont possibles :
- **à la main** : traduis chaque information en une équation sur les cases, et résous-les dans le bon ordre (une case après l'autre) ;
- **par le code** : cherche, parmi les tableaux d'entiers qui ont les bonnes marges, ceux qui vérifient toutes les informations (attention : les comparaisons de fractions se font avec `math.isclose` ou avec `fractions.Fraction`, pas avec `==` sur des flottants).

Puis calcule b) $P(\text{appli} \mid \text{achat ou abandon})$, en fraction irréductible.

La vérification affiche, pour ton tableau, les informations qu'il respecte (✓) ou non (✗), puis vérifie tes réponses.

Dans ta copie : la solution est-elle unique ? Quelles informations suffiraient à la trouver si l'on retirait l'information 6 ? Le canal et le résultat sont-ils indépendants ?

In [ ]:
TABLE_28 = ...         # a) [[app-purchase, app-abandoned, app-nothing], [site-purchase, site-abandoned, site-nothing]]
answer_M03_28b = ...   # b) an irreducible fraction, as text

In [ ]:
with checked("M03.28 indices"):
    if TABLE_28 is ...:
        raise NotImplementedError("remplis TABLE_28")
    table = np.asarray(TABLE_28, dtype=float)
    if table.shape != (2, 3):
        print("❌ TABLE_28 doit avoir 2 lignes (appli, site) et 3 colonnes (achat, abandon, rien).")
    else:
        rows, cols, n = table.sum(axis=1), table.sum(axis=0), table.sum()
        clues = [("600 visites", n == 600),
                 ("P(appli) = 0,4", math.isclose(rows[0] / n, 0.4)),
                 ("P(achat) = 0,5", math.isclose(cols[0] / n, 0.5)),
                 ("P(achat | appli) = 0,625", rows[0] > 0 and math.isclose(table[0, 0] / rows[0], 0.625)),
                 ("P(appli | abandon) = 1/6", cols[1] > 0 and math.isclose(table[0, 1] / cols[1], 1 / 6)),
                 ("P(rien | site) = 1/6", rows[1] > 0 and math.isclose(table[1, 2] / rows[1], 1 / 6))]
        print("  ".join(f"{'✓' if ok else '✗'} {name}" for name, ok in clues))
wbm.check("M03.28a", TABLE_28)
wbm.check("M03.28b", answer_M03_28b)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu lire dans un tableau des probabilités jointes, marginales et conditionnelles, décider de l'indépendance, et utiliser les probabilités totales ?
2. Sais-tu écrire chaque mesure d'une matrice de confusion comme une probabilité conditionnelle, et dire comment elle change avec la prévalence et le seuil ?
3. Sais-tu calculer une AUC par les paires, une average precision et un score de Brier, et dire ce que chacun mesure ?

**La suite** : le chapitre 3 de workbookIA (probabilités et métriques de classification), puis M04, la règle de Bayes et la loi Beta.